# 02 - Feature engineering (strictly pre-loan behavior)

Every behavioral feature for a loan uses **only transactions dated before that loan's grant date**
(`trans.date < loan.date`). Post-loan activity is never touched, so no leakage is possible.
Output: `../data/loan_features.csv` - one row per loan (682).

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA_DIR = ROOT / "data"

dfs = {n: pd.read_csv(DATA_DIR / f"{n}.csv", sep=";", low_memory=False) for n in ["loan", "trans", "client", "disp"]}
loan, trans, client, disp = (dfs[n] for n in ["loan", "trans", "client", "disp"])

trans["parsed_date"] = pd.to_datetime(trans["date"].astype(str), format="%y%m%d")
loan["parsed_date"] = pd.to_datetime(loan["date"].astype(str), format="%y%m%d")
loan["target"] = loan["status"].map({"A": 0, "C": 0, "B": 1, "D": 1})
print(loan.shape, trans.shape)

(682, 9) (1056320, 11)


In [2]:
WITHDRAW = {"VYDAJ", "VYBER"}
prior_by_acct = {acct: g.sort_values("parsed_date") for acct, g in trans.groupby("account_id")}


def behavior_for(account_id, grant_date):
    g = prior_by_acct.get(account_id)
    prior = g[g["parsed_date"] < grant_date] if g is not None else []
    if len(prior) == 0:
        return dict.fromkeys(["n_txn", "n_deposit", "n_withdrawal", "sum_deposit", "sum_withdrawal", "avg_amount", "std_amount", "avg_balance", "min_balance", "max_balance", "balance_at_loan", "months_active", "txn_per_month"], 0.0)
    dep = prior.loc[prior["type"] == "PRIJEM", "amount"]
    wit = prior.loc[prior["type"].isin(WITHDRAW), "amount"]
    months = max((grant_date - prior["parsed_date"].min()).days / 30.44, 1 / 30.44)
    return {
        "n_txn": len(prior),
        "n_deposit": int((prior["type"] == "PRIJEM").sum()),
        "n_withdrawal": int(prior["type"].isin(WITHDRAW).sum()),
        "sum_deposit": float(dep.sum()),
        "sum_withdrawal": float(wit.sum()),
        "avg_amount": float(prior["amount"].mean()),
        "std_amount": float(prior["amount"].std(ddof=0)),
        "avg_balance": float(prior["balance"].mean()),
        "min_balance": float(prior["balance"].min()),
        "max_balance": float(prior["balance"].max()),
        "balance_at_loan": float(prior["balance"].iloc[-1]),
        "months_active": float(months),
        "txn_per_month": float(len(prior) / months),
    }


feat = pd.DataFrame(
    [{"loan_id": l.loan_id, **behavior_for(l.account_id, l.parsed_date)} for l in loan.itertuples()]
)
print(feat.shape)
print("loans with zero prior txns:", int((feat['n_txn'] == 0).sum()))
print(feat.describe().round(1).to_string())

(682, 14)
loans with zero prior txns: 0
       loan_id  n_txn  n_deposit  n_withdrawal  sum_deposit  sum_withdrawal  avg_amount  std_amount  avg_balance  min_balance  max_balance  balance_at_loan  months_active  txn_per_month
count    682.0  682.0      682.0         682.0        682.0           682.0       682.0       682.0        682.0        682.0        682.0            682.0          682.0          682.0
mean    6172.5   80.2       32.0          48.1     385871.8        344055.4      9488.7     10916.7      43606.8        656.2      89016.2          42550.8           13.1            5.8
std      682.6   44.8       15.1          31.6     265602.1        259830.8      4445.5      4631.2      13481.9       1845.9      34245.5          22742.2            5.4            1.7
min     4959.0    2.0        2.0           0.0      10100.0             0.0      1566.9      1919.4       5350.0     -17030.4      10100.0          -1571.5            3.4            0.6
25%     5577.5   45.0       21

In [3]:
bn = client["birth_number"].astype(str).str.zfill(6)
demo = pd.DataFrame({
    "client_id": client["client_id"],
    "gender": (bn.str[2:4].astype(int) > 50).map({True: "Female", False: "Male"}),
    "age": 1999 - (bn.str[:2].astype(int) + 1900),
    "district_id": client["district_id"],
})
owner = disp.loc[disp["type"] == "OWNER", ["account_id", "client_id"]]

features = (
    loan.merge(owner, on="account_id", how="left")
    .merge(feat, on="loan_id", how="left")
    .merge(demo, on="client_id", how="left")
)
features["gender_bin"] = (features["gender"] == "Female").astype(int)

behav_cols = [c for c in feat.columns if c != "loan_id"]
features[behav_cols] = features[behav_cols].fillna(0)
features.to_csv(DATA_DIR / "loan_features.csv", index=False)
print(features.shape)
print(features[["loan_id", "amount", "duration", "target", "n_txn", "avg_balance", "balance_at_loan", "age", "gender"]].head(3).to_string())

(682, 27)
   loan_id  amount  duration  target  n_txn   avg_balance  balance_at_loan  age  gender
0     5314   96396        12       1      4  12250.000000          20100.0   52  Female
1     5316  165960        36       0     37  52083.859459          52208.9   31    Male
2     6863  127080        60       0     24  30060.954167          20272.8   63    Male


In [4]:
assert owner["account_id"].is_unique, "duplicate OWNER per account would fan out joins"
assert features["client_id"].notna().all(), "loan without OWNER client"
assert features["account_id"].isin(trans["account_id"]).all(), "orphan loan account_id"
assert len(features) == 682, len(features)
assert features[behav_cols].isnull().sum().sum() == 0
for l in loan.sample(5, random_state=42).itertuples():
    g = prior_by_acct[l.account_id]
    expected = g.loc[g["parsed_date"] < l.parsed_date, "balance"]
    row = features.loc[features["loan_id"] == l.loan_id].iloc[0]
    if len(expected):
        assert row["balance_at_loan"] == expected.iloc[-1]
        assert (g.loc[g["parsed_date"] < l.parsed_date, "parsed_date"] < l.parsed_date).all()
    else:
        assert row["n_txn"] == 0
neg = int((features["min_balance"] < 0).sum())
print(f"loans with overdraft (min_balance<0): {neg}")
print("feature checks passed (incl. pre-loan leakage spot-check)")

loans with overdraft (min_balance<0): 27
feature checks passed (incl. pre-loan leakage spot-check)
